# Module 3 — Class 3: Feature Engineering

**Lecture reference:** `MODULE_3_OLIST_REWRITE.md` § Class 3-3

**Today:** build the features that actually predict `is_late`. Date decomposition, ratios, and the killer feature: **`distance_km`** between customer and seller.

## Why this matters today

The columns you have right now (price, freight, payment_type) are necessary but not sufficient. The features that predict `is_late` are *derived*: how far did the parcel travel? How many days from purchase to estimate? Was the order placed on a weekend?

Good feature engineering beats fancy models on tabular data, every time.

In [ ]:
import pandas as pd, numpy as np

df = pd.read_parquet('orders_step2.parquet')
print(df.shape)
df.head(3)

## Section 1 — The headline target: `is_late`

In [ ]:
# Filter rows that have a delivery date (already done in step 1, but defensive)
df = df.dropna(subset=['order_delivered_customer_date',
                       'order_estimated_delivery_date'])
df['is_late'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype(int)

print('is_late rate:', df['is_late'].mean().round(3))   # ≈ 0.07 — imbalanced!

## Section 2 — Date-based features

In [ ]:
df['delivery_days'] = (df['order_delivered_customer_date']
                       - df['order_purchase_timestamp']).dt.days
df['estimate_days'] = (df['order_estimated_delivery_date']
                       - df['order_purchase_timestamp']).dt.days
df['days_buffer']   = df['estimate_days'] - df['delivery_days']  # negative = late

df['purchase_hour']      = df['order_purchase_timestamp'].dt.hour
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['purchase_month']     = df['order_purchase_timestamp'].dt.month
df['is_weekend']         = (df['purchase_dayofweek'] >= 5).astype(int)

df[['delivery_days','estimate_days','days_buffer',
    'purchase_hour','purchase_dayofweek','is_weekend','is_late']].head()

## Section 3 — The killer feature: Haversine distance

Brazilian distances are huge. Customer in Manaus, seller in São Paulo = 4000 km. Distance is THE strongest signal for late delivery.

In [ ]:
# Aggregate geolocation by zip prefix (faster than per-row joins)
geo = pd.read_csv('olist_data/olist_geolocation_dataset.csv')
geo_lookup = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lon=('geolocation_lng', 'mean'),
).reset_index()
geo_lookup.head()

In [ ]:
# Merge customer + seller lat/lon
customers = pd.read_csv('olist_data/olist_customers_dataset.csv')
items     = pd.read_csv('olist_data/olist_order_items_dataset.csv')
sellers   = pd.read_csv('olist_data/olist_sellers_dataset.csv')

# Customer lat/lon
df = df.merge(customers[['customer_id', 'customer_zip_code_prefix']],
              on='customer_id', how='left')
df = df.merge(geo_lookup.rename(columns={'lat':'cust_lat', 'lon':'cust_lon'}),
              left_on='customer_zip_code_prefix',
              right_on='geolocation_zip_code_prefix', how='left')

# Seller lat/lon (one seller per order — pick the first item's seller)
first_seller = (items.sort_values(['order_id', 'order_item_id'])
                     .drop_duplicates('order_id')[['order_id', 'seller_id']])
df = df.merge(first_seller, on='order_id', how='left')
df = df.merge(sellers[['seller_id', 'seller_zip_code_prefix']],
              on='seller_id', how='left')
df = df.merge(geo_lookup.rename(columns={'lat':'seller_lat', 'lon':'seller_lon',
                                         'geolocation_zip_code_prefix':'sz'}),
              left_on='seller_zip_code_prefix', right_on='sz', how='left')

In [ ]:
# Vectorised Haversine
R = 6371   # Earth radius in km

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine(df['cust_lat'].values, df['cust_lon'].values,
                              df['seller_lat'].values, df['seller_lon'].values)
print(df['distance_km'].describe().round(1))

## Section 4 — Quick sanity check: does distance correlate with `is_late`?

In [ ]:
# Group distance into 5 bins, look at is_late rate per bin
df['distance_bin'] = pd.cut(df['distance_km'], bins=[0, 100, 500, 1000, 2000, 10000])
df.groupby('distance_bin', observed=True)['is_late'].mean().round(3)

Expect a clear monotonic increase. Far → late more often. **That's a real signal.**

## Section 5 — Save Stage-3

In [ ]:
df.to_parquet('orders_step3.parquet', index=False)
print('orders_step3.parquet:', df.shape)

## ✅ Quick Check

1. Why filter out rows where `order_delivered_customer_date` is NaN before computing `is_late`?  
2. Why is Haversine the right distance, not Euclidean?  
3. Name two features you could engineer from `seller_id` alone (joining order history).

## 📝 Homework (Bronze)

Run all cells. Submit `orders_step3.parquet`.  
Add a markdown cell: which feature do you think will most predict `is_late`? Justify in 2 sentences.

## 🔥 Homework (Gold)

Engineer **at least 2 additional features** beyond what's in this notebook. Examples:
- `freight_per_item = total_freight / num_items`
- `seller_avg_delivery_days` (avg historical delivery time per seller)
- `is_high_volume_state` (state with > 5000 orders)

Justify each in 2 sentences. Save as `orders_step3_gold.parquet`.